# Part 3-D — Experimental Evaluation: Prompting Strategies

**Compares:**
1. Zero-shot prompting
2. Paper's multi-step human-like strategy (MAPS)

**Dataset:** WMT16 en-tr test split (200 samples, seed=42)  
**Model:** Qwen 2.5 7B Instruct (4-bit quantized)  
**Metric:** COMET (wmt22-comet-da)

> Results are saved to `results/` so Part 5 can load them without re-running inference.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import json
import pandas as pd
import matplotlib.pyplot as plt

from modules.dataset import load_wmt16, get_samples
from modules.model import load_model
from modules.translation import (
    translate_zero_shot,
    translate_paper_strategy,
    load_cached_translations,
)
from modules.evaluation import load_comet_model, compute_comet

os.makedirs('../results', exist_ok=True)

## 3-D.1 Load Data

In [ ]:
N_SAMPLES = 200
SEED = 42

test_ds = load_wmt16(split='test')
en_sentences, tr_references = get_samples(test_ds, n=N_SAMPLES, seed=SEED)

print(f"\nEvaluation set: {len(en_sentences)} pairs")
print(f"  EN[0]: {en_sentences[0]}")
print(f"  TR[0]: {tr_references[0]}")

## 3-D.2 Load Model

In [ ]:
model, tokenizer = load_model(quantize=True)

## 3-D.3 Strategy A — Zero-Shot Translation

In [ ]:
ZERO_SHOT_PATH = '../results/zero_shot_translations.json'

if os.path.exists(ZERO_SHOT_PATH):
    print(f"Loading cached zero-shot translations from {ZERO_SHOT_PATH}")
    zero_shot_translations = load_cached_translations(ZERO_SHOT_PATH)
else:
    zero_shot_translations = translate_zero_shot(
        model, tokenizer, en_sentences,
        direction='en->tr',
        save_path=ZERO_SHOT_PATH
    )

print(f"\nSample zero-shot outputs:")
for i in range(3):
    print(f"  EN : {en_sentences[i]}")
    print(f"  REF: {tr_references[i]}")
    print(f"  HYP: {zero_shot_translations[i]}")
    print()

## 3-D.4 Strategy B — Paper's Multi-Step Strategy

In [ ]:
PAPER_PATH = '../results/paper_strategy_translations.json'

if os.path.exists(PAPER_PATH):
    print(f"Loading cached paper-strategy translations from {PAPER_PATH}")
    paper_translations = load_cached_translations(PAPER_PATH)
else:
    paper_translations = translate_paper_strategy(
        model, tokenizer, en_sentences,
        direction='en->tr',
        save_path=PAPER_PATH
    )

print(f"\nSample paper-strategy outputs:")
for i in range(3):
    print(f"  EN : {en_sentences[i]}")
    print(f"  REF: {tr_references[i]}")
    print(f"  HYP: {paper_translations[i]}")
    print()

## 3-D.5 COMET Evaluation

In [ ]:
comet_model = load_comet_model()

In [ ]:
print("Computing COMET — Zero-shot...")
zs_result = compute_comet(en_sentences, zero_shot_translations, tr_references, comet_model)

print("Computing COMET — Paper strategy...")
ps_result = compute_comet(en_sentences, paper_translations, tr_references, comet_model)

results = {
    'zero_shot': zs_result['system_score'],
    'paper_strategy': ps_result['system_score'],
}

# Save for Part 5
with open('../results/part3_comet_scores.json', 'w') as f:
    json.dump(results, f, indent=2)

print("\n" + "=" * 45)
print(f"{'Approach':<25} {'COMET':>10}")
print("=" * 45)
for name, score in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{name:<25} {score:>10.4f}")
print("=" * 45)

## 3-D.6 Per-Sentence Score Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(zs_result['scores'], bins=30, alpha=0.6, label=f"Zero-shot (μ={zs_result['system_score']:.4f})",
        color='steelblue', edgecolor='white')
ax.hist(ps_result['scores'], bins=30, alpha=0.6, label=f"Paper strategy (μ={ps_result['system_score']:.4f})",
        color='darkorange', edgecolor='white')

ax.set_xlabel('COMET Score', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Per-Sentence COMET Score Distribution — Zero-shot vs Paper Strategy', fontsize=13)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/part3_comet_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/part3_comet_distribution.png')

## 3-D.7 Discussion

The paper strategy prompts the model to explicitly:
1. Analyze terminology and cultural context before translating
2. Produce a draft
3. Self-review and refine

Expected outcomes:
- **Higher COMET for paper strategy** on complex sentences with domain-specific vocabulary or idioms
- **Similar performance** on simple, short sentences where zero-shot is sufficient
- **Higher latency** for paper strategy (~2-3x longer output per sentence)

The per-sentence distribution plot reveals whether improvements are uniform or concentrated on specific sentence types.